# HazNet — G 데이터로 학습한 가벼운 위험 분류기, 실제 위험 영상으로 검증

**묻는 것**: 우리 G 가 만든 합성 위험 영상(정답 = 하드 판정기)만으로 학습한 작은 CNN 이 **실제 위험 영상**에서도 통하는가.

| 단계 | 무엇 | 코드 |
|---|---|---|
| 데이터 | G 3종: 휘도 `w2s` · 적색 `r4z2` · 줄무늬 `s4dc`(직선 정지·반전·흐름) + `s5pat`(직선·동심원·방사형·나선·처프·체커·물결) + 깨끗한 바탕 | `make_gdr_dataset.py` → `hazcls_cache.py synth` |
| 라벨 | 클립마다 휘도 / 적색 / 무늬 위험 (하드 판정기) + 위험 픽셀 지도(튜브) | manifest `judge`, npz `tubes` |
| 모델 | 갈래마다 합성곱 **5층**: 시간 갈래(점멸) + 공간 갈래(무늬) → 위험 지도 → 면적 → 클립 판정. 매개변수 약 4.7만 개 | `src/lumi_synth/hazcls.py` |
| 증강 | 좌우·상하·시간 뒤집기, 회전 ±25°, 화질(축소→확대·흐림·블록 양자화·잡음), 약한 밝기 이득 | `hazcls.augment` |
| 검증 | 합성 val(다른 영화 바탕) + **실제 val**(Kaya 16개 + South 안전 298개) | |
| 조기 종료 | 감시 손실이 `patience` epoch 동안 줄지 않으면 멈추고 가장 좋았던 epoch 로 되돌림 | `train_hazcls.py` |
| 시험 | **실제 test**: South 위험 163개(시험 전용) + Kaya 6개 + South 안전 594개 — 학습·조기 종료에 쓰지 않음 | |

⚠️ 모든 영상은 광과민성 위험 자극이다. 이 노트북은 **숫자와 그래프만** 보여 주고 영상·프레임을 표시하지 않는다.

캐시가 없으면 아래 '0. 캐시 만들기'를 먼저 (터미널에서 한 번만, 수십 분).

## 0. 캐시 만들기 (한 번만 — 이미 있으면 건너뜀)

In [ ]:
import subprocess, sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
PY = sys.executable
C = ROOT / 'data' / 'hazcls'
def sh(*a):
    print('$', ' '.join(map(str, a)))
    r = subprocess.run([PY, '-u', *map(str, a)], cwd=ROOT, capture_output=True, text=True, encoding='utf-8', errors='replace')
    print(r.stdout[-1500:], r.stderr[-1500:])

jobs = {
    'train':    ['scripts/hazcls_cache.py', 'synth', '--name', 'train', '--dirs', 'data/gdr/v8/train', 'data/gdr/v9s/train'],
    'val':      ['scripts/hazcls_cache.py', 'synth', '--name', 'val', '--dirs', 'data/gdr/v8/val', 'data/gdr/v9s/val'],
    'val_rule': ['scripts/hazcls_cache.py', 'synth', '--name', 'val_rule', '--dirs', 'data/gdr/v8/val_rule'],
    'real':     ['scripts/hazcls_cache.py', 'real', '--name', 'real'],
    'real_pair':['scripts/hazcls_cache.py', 'real', '--pairs', '--name', 'real_pair'],
}
for name, a in jobs.items():
    if (C / name / 'tem.npy').exists():
        print('있음:', name)
    else:
        sh(*a)

## 1. 데이터 구성 확인

In [ ]:
import numpy as np, collections
for n in ('train', 'val', 'val_rule'):
    y = np.load(C / n / 'y.npy'); rows = [json.loads(l) for l in open(C / n / 'rows.jsonl', encoding='utf-8')]
    print(f"{n:9s} {len(y):5d}개 · 위험 휘도 {y[:,0].sum():4d} · 적색 {y[:,1].sum():4d} · 무늬 {y[:,2].sum():4d} · 안전 {(y.sum(1)==0).sum():4d}",
          '· 생성원', dict(collections.Counter((r['ckpt'] or r['generator'] or '바탕') if r['hazard'] else '바탕' for r in rows)))
items = [json.loads(l) for l in open(C / 'real' / 'items.jsonl', encoding='utf-8')]
print('실제:', dict(collections.Counter((i['source'], i['split'], '위험' if i['any'] else '안전') for i in items if i.get('n_win'))))

## 2. 학습 (조기 종료)

- `monitor='both'`: 합성 val 손실 + 실제 val 균형 손실 — 사용자 요청대로 실제 영상으로 검증
- `monitor='synth'`: 합성 val 만 — 비교용. **실제 val 양성은 Kaya 뿐이라 'Kaya 처럼 생긴 영상' 을 배워도 실제 val 손실이 줄 수 있다** (동작 확인에서 거의 학습 안 한 모델이 실제 val AUC 0.92, 같은 South 안 test AUC 0.50)
- 이미 학습한 run 이름이면 멈춘다(`overwrite=True` 로 덮어씀)

In [ ]:
sys.path.insert(0, str(ROOT / 'scripts')); sys.path.insert(0, str(ROOT / 'src'))
import importlib, train_hazcls as th
importlib.reload(th)
RUN = 'nb_a'
args = th.parse(['--run', RUN, '--monitor', 'both', '--max-epochs', '40', '--patience', '4'])
if (C / f'{RUN}.pt').exists():
    print('이미 학습함 → 기록만 읽음 (다시 하려면 --overwrite 를 넣거나 RUN 이름을 바꾸기)')
else:
    res = th.run(args)

## 3. 학습 곡선 (점선 = 조기 종료로 고른 epoch)

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display
display(Image(str(ROOT / 'checks' / 'hazcls' / f'{RUN}_curves.png')))
hist = [json.loads(l) for l in open(ROOT / 'checks' / 'hazcls' / f'{RUN}_log.jsonl', encoding='utf-8')]
for h in hist:
    print(h['epoch'], '학습', h['train_loss'], '· 합성 val', h['val']['loss'], '· 실제 val', h.get('real_val', {}).get('bal_loss'),
          'AUC', h.get('real_val', {}).get('auc'), '★' if h.get('best') else '')

## 4. 결과 보고서 (실제 test · 판정기 비교 · Kaya 보정 전후 짝)

In [ ]:
txt = (ROOT / 'checks' / 'hazcls' / f'{RUN}.md').read_text(encoding='utf-8')
display(Markdown(txt[txt.index('## 검증'):]))

## 5. 실제 test 점수 분포 (South 위험 종류별 · 안전)

In [ ]:
rows = [json.loads(l) for l in open(ROOT / 'checks' / 'hazcls' / f'{RUN}_test_items.jsonl', encoding='utf-8')]
groups = {'South 안전': [r['p_any'] for r in rows if r['source']=='south' and not r['any']],
          'South 깜빡임': [r['p_any'] for r in rows if r['source']=='south' and r['lum']==1],
          'South 적색': [r['p_any'] for r in rows if r['source']=='south' and r['red']==1],
          'South 패턴': [r['p_any'] for r in rows if r['source']=='south' and r['pattern']==1],
          'Kaya': [r['p_any'] for r in rows if r['source']=='kaya']}
plt.rcParams['font.family'] = 'Malgun Gothic'; plt.rcParams['axes.unicode_minus'] = False
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.boxplot(list(groups.values()), labels=[f'{k}\n({len(v)})' for k, v in groups.items()], showfliers=True)
ax.set_ylabel('HazNet 위험 확률 (영상 단위)'); ax.axhline(0.5, ls=':', c='gray')
plt.tight_layout(); plt.show()

## 6. 해석 메모

- 합성 val 은 잘 맞히는데 실제 test 에서 무너지면: G 데이터가 실제 영상의 모습(장면 전환·자막·압축·GIF 팔레트)을 덮지 못한다는 뜻 → 바탕을 실제 영상류로 넓히거나, South 안전 영상(위험 아님)을 음성으로 학습에 넣는 실험(`--real-neg-train`, 아직 없음)이 다음 단계.
- 패턴 종류의 민감도는 데이터셋 라벨(South 'pattern')과 우리 판정 기준(6쌍 · 1~10 cpd · 면적 40%)의 정의 차이도 섞여 있다 — 판정기 둘의 같은 영상 결과와 함께 본다.